# 🎓 LoRA Fine-tuning v2 (Tutorial-based / KcBERT)

**보정된 UnSmile + 수집된 게임 음성채팅 데이터**로 학습합니다.

- **모델**: `beomi/kcbert-base`
- **메트릭**: `abuse_recall` (통일)
- **데이터**: UnSmile 보정 14,690 + 수집 518 = **15,208건**

In [1]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA L40S


In [2]:
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./output/lora_tutorial_kcbert_v2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 통일된 하이퍼파라미터
EPOCHS, BATCH_SIZE, LEARNING_RATE = 10, 32, 2e-4
MAX_LENGTH = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
unsmile_train = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
collected_df = pd.read_csv("../../0_Data_Collection/datasets/train_collected.tsv", sep='\t')

train_df = pd.concat([unsmile_train, collected_df], ignore_index=True)
print(f"✅ Train: {len(train_df)}건 (보정 {len(unsmile_train)} + 수집 {len(collected_df)})")
print(f"Valid: {len(valid_df)}건")

✅ Train: 15208건 (보정 14690 + 수집 518)
Valid: 3663건


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

Map:   0%|          | 0/15208 [00:00<?, ? examples/s]

Map:   0%|          | 0/3663 [00:00<?, ? examples/s]

In [5]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification")
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=["query", "key", "value"], bias="none")
model = get_peft_model(base_model, peft_config).to(DEVICE)
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 892,426 || all params: 109,818,644 || trainable%: 0.8126


In [6]:
# 통일된 compute_metrics (abuse_recall 포함)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    _, clean_r, clean_f1, _ = precision_recall_fscore_support(labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    return {'lrap': label_ranking_average_precision_score(labels, predictions), 'abuse_recall': abuse_r, 'abuse_f1': abuse_f1, 'clean_recall': clean_r, 'clean_f1': clean_f1}

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="abuse_recall", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

In [8]:
print("🚀 v2 학습 시작...")
trainer.train()
print("학습 완료!")

🚀 v2 학습 시작...


Epoch,Training Loss,Validation Loss,Lrap,Abuse Recall,Abuse F1,Clean Recall,Clean F1
1,0.334000,0.271010,0.612739,0.000000,0.000000,0.418280,0.512516
2,0.214900,0.168824,0.825756,0.491634,0.588145,0.677419,0.707071
3,0.152000,0.142744,0.849340,0.552124,0.626735,0.676344,0.718857
4,0.135200,0.136326,0.853952,0.621622,0.654472,0.632258,0.705036
5,0.126400,0.129937,0.864762,0.606178,0.659664,0.726882,0.744904
6,0.121000,0.128146,0.866191,0.635779,0.663533,0.731183,0.741953
7,0.115100,0.127573,0.867352,0.671815,0.673983,0.705376,0.738323
8,0.114400,0.125793,0.868165,0.670528,0.676184,0.726882,0.742857
9,0.108400,0.125176,0.870118,0.655084,0.677312,0.744086,0.744887
10,0.109800,0.125299,0.870582,0.652510,0.672860,0.733333,0.746171


학습 완료!


In [9]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
print("✅ v2 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")

✅ v2 모델 저장 완료!


  eval_loss: 0.1276
  eval_lrap: 0.8674
  eval_abuse_recall: 0.6718
  eval_abuse_f1: 0.6740
  eval_clean_recall: 0.7054
  eval_clean_f1: 0.7383
  eval_runtime: 2.5107
  eval_samples_per_second: 1458.9350
  eval_steps_per_second: 11.5500
  epoch: 10.0000
